# **Corrupt Codex (SOLUTION)**

<div style="font-size: 14px; color: #6e8192; line-height: 1.5;">
  <div style="display: flex; align-items: center; gap: 5px; margin-bottom: 5px;">
    <span style="font-size: 18px; color: #6e8192;">🎯</span>
    <span>MI National Olympiad, Summer Selection</span>
  </div>
  <div style="display: flex; align-items: center; gap: 5px;">
    <span style="font-size: 18px; color: #6e8192;">🧠</span>
    <span>Machine Learning: Solution Notebook</span>
  </div>
  <div style="display: flex; align-items: center; gap: 5px;">
    <span style="font-size: 18px; color: #6e8192;">🏆</span>
    <span>100 points</span>
  </div>
  <div style="display: flex; align-items: center; gap: 5px;">
    <span style="font-size: 18px; color: #6e8192;">🗓️</span>
    <span>May 2026</span>
  </div>
</div>

> **Note:** this notebook is the **organizer/educational reference** version. It contains the full test set's true labels (loading `_target.csv` with the `is_memorized` field), and evaluates all membership inference strategies via ROC AUC. These are **not provided** in the contestant (RAW) notebook: the contestant only receives the 12-feature test points (`test.csv`), 50 labeled calibration samples (`calibration.csv`), and the trained network weights (`net_weights.pt`). The SOLUTION aims to allow organizers to calibrate the scoring curve and make the contestant pipeline comparable to the reference.

## **Setup and imports**

In [1]:
# ═══════════════════════════════════════════════════════════════════
# DO NOT MODIFY: with this seed you get the same result every
# rerun. The server is deterministic.
# ═══════════════════════════════════════════════════════════════════
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

# --- Reproducibility: tie all randomness to a single seed ---
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# GPU is not necessary for the task, but can be used.
device = "cuda" if torch.cuda.is_available() else "cpu"
print('device:', device)

device: cuda


## **Download dataset (organizer)**

The 3 contestant files (`calibration.csv`, `test.csv`, `net_weights.pt`) are downloaded with the same Drive FILE_IDs as in the RAW notebook; there is no privileged data here.

The `organizer/data/_target.csv` (the full 5000-row GT) is **organizer-only** and we assume it is locally available in the repo structure. If missing, the cell clearly indicates.

In [2]:
import os
import subprocess
import sys
from pathlib import Path

DATA_DIR = Path("data")
ORG_DIR  = Path("organizer/data")
DATA_DIR.mkdir(exist_ok=True)
ORG_DIR.mkdir(parents=True, exist_ok=True)

# 3 contestant files with the same FILE_IDs as in the RAW notebook.
# 4. (_target.csv) ORGANIZER-ONLY: separate organizer Drive ID, DO NOT SHARE.
FILES = {
    "calibration.csv": ("1pwdxlA-aJ59eb8P3rlqRKjPKp4SS-Ugp", DATA_DIR),
    "test.csv":        ("1fkQ3IpdUfhJY46iPCShKOGjpd8W-rFmE", DATA_DIR),
    "net_weights.pt":  ("1Lx1t3OYgvhDSU573asPcW8FGP90V18aX", DATA_DIR),
    "_target.csv":     ("1jPB1eUrjGNQDzRgHbRbnfyN9NXckcXyZ",                          ORG_DIR),
}
missing = [(name, fid, dst) for name, (fid, dst) in FILES.items()
           if not (dst / name).exists()]

if not missing:
    print("All files present, skipping download.")
else:
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    for name, file_id, dst in missing:
        gdown.download(id=file_id, output=str(dst / name), quiet=False)

for name, (_, dst) in FILES.items():
    p = dst / name
    print(f"  {p}: {'OK' if p.exists() else 'MISSING'}")

Downloading...
From: https://drive.google.com/uc?id=1pwdxlA-aJ59eb8P3rlqRKjPKp4SS-Ugp
To: c:\Users\raian\source\repos\AI\IOAI_prep\hungary\2026\model_interpretation\data\calibration.csv
100%|██████████| 6.91k/6.91k [00:00<?, ?B/s]
Downloading...
From: https://drive.google.com/uc?id=1fkQ3IpdUfhJY46iPCShKOGjpd8W-rFmE
To: c:\Users\raian\source\repos\AI\IOAI_prep\hungary\2026\model_interpretation\data\test.csv
100%|██████████| 669k/669k [00:00<00:00, 6.42MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Lx1t3OYgvhDSU573asPcW8FGP90V18aX
To: c:\Users\raian\source\repos\AI\IOAI_prep\hungary\2026\model_interpretation\data\net_weights.pt
100%|██████████| 281k/281k [00:00<00:00, 4.46MB/s]
Downloading...
From: https://drive.google.com/uc?id=1jPB1eUrjGNQDzRgHbRbnfyN9NXckcXyZ
To: c:\Users\raian\source\repos\AI\IOAI_prep\hungary\2026\model_interpretation\organizer\data\_target.csv
100%|██████████| 689k/689k [00:00<00:00, 4.60MB/s]


  data\calibration.csv: OK
  data\test.csv: OK
  data\net_weights.pt: OK
  organizer\data\_target.csv: OK


## **The trained model**

12 → 256 → 256 → 1 feed-forward network, with sin activations (SIREN style). We load the weights back into the `Net` class using the initialization used during training.

In [3]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(12, 256)
        self.fc2 = nn.Linear(256, 256)
        self.fc3 = nn.Linear(256, 1)

        # Init range as used during training (SIREN-like).
        with torch.no_grad():
            self.fc1.weight.uniform_(-1 / 12, 1 / 12)
            bound2 = (6 / 256) ** 0.5 / 30
            self.fc2.weight.uniform_(-bound2, bound2)
            bound3 = (6 / 256) ** 0.5
            self.fc3.weight.uniform_(-bound3, bound3)

    def forward(self, x):
        x = torch.sin(30 * self.fc1(x))
        x = torch.sin(30 * self.fc2(x))
        return self.fc3(x)

model = Net().to(device)
model.load_state_dict(torch.load('data/net_weights.pt', map_location=device))
model.eval()
print('parameters:', sum(p.numel() for p in model.parameters()))

parameters: 69377


## **Load data**

`_target.csv` is **organizer-only**: the true `is_memorized` (1 = fraudulent / wrong label, 0 = legit) status of all 5000 samples. The contestant only sees `test.csv` and the 50-row `calibration.csv`.

In [4]:
calibration_df = pd.read_csv('data/calibration.csv')
test_df        = pd.read_csv('data/test.csv')
target_df      = pd.read_csv('organizer/data/_target.csv')   # ORGANIZER: ground truth

calibration = torch.tensor(calibration_df.values, dtype=torch.float32).to(device)
test        = torch.tensor(test_df.values,        dtype=torch.float32).to(device)

X_calibration = calibration[:, :-1]
y_calibration = calibration[:, -1]
X_test        = test  # test.csv has no target column

is_memorized = torch.tensor(target_df['target'].values, dtype=torch.float32)

print(f'calibration: {X_calibration.shape}, positive ratio: {y_calibration.mean():.3f}')
print(f'test:        {X_test.shape}')
print(f'is_memorized positive ratio (GT): {is_memorized.mean():.3f}')

calibration: torch.Size([50, 12]), positive ratio: 0.340
test:        torch.Size([5000, 12])
is_memorized positive ratio (GT): 0.300


## **Reproduction of the evaluation**

The platform's server-side evaluation logic: it calculates a ROC AUC based on the submitted `score` and the true `is_memorized` column, then converts it to points with a piecewise linear curve:

| AUC | Points |
|---:|---:|
| ≤ 0.50 | 0 |
| 0.70 | 10 |
| 0.80 | 25 |
| 0.85 | 50 |
| 0.90 | 70 |
| 0.95 | 90 |
| 1.00 | 100 |

Linear interpolation between anchor points. The curve is mostly convex; every additional 0.05 AUC improvement yields significantly more points in the upper range.

In [5]:
# Scoring curve, exactly the same as in the server-side evaluator.
ANCHORS = [(0.50, 0), (0.70, 10), (0.80, 25), (0.85, 50),
           (0.90, 70), (0.95, 90), (1.00, 100)]

def auc_to_points(auc: float) -> float:
    if auc <= ANCHORS[0][0]:
        return ANCHORS[0][1]
    for i in range(1, len(ANCHORS)):
        x0, y0 = ANCHORS[i-1]
        x1, y1 = ANCHORS[i]
        if auc <= x1:
            return y0 + (y1 - y0) * (auc - x0) / (x1 - x0)
    return ANCHORS[-1][1]

def evaluate(score, label=''):
    # The server-side scorer computes raw AUC, does not flip automatically.
    # If AUC < 0.5, your score has the wrong sign; try `-score`.
    auc = roc_auc_score(is_memorized.cpu(), score)
    pts = auc_to_points(auc)
    if label:
        print(f'{label:42s}  AUC = {auc:.4f}  -> {pts:5.1f} points')
    return auc, pts

## **Warm-up: model `forward` logit as score**

The simplest baseline: the submitted score is the model's logit. Since the network was trained with higher sample weights on the flipped (wrongly labeled) samples, their logits are often closer to 0 (less confidently classified), while true negatives are deep in the negative range.

In [6]:
with torch.no_grad():
    logits = model(X_test).squeeze().cpu()

evaluate(logits, 'Baseline: logit confidence')

Baseline: logit confidence                  AUC = 0.7423  ->  16.3 points


(0.7422547619047619, 16.338214285714287)

## **Strategies**

### 1. **Perturbation Flip Rate**

Classical *adversarial* membership inference signal. The network learns a sharp decision boundary around memorized points; adding small Gaussian noise often flips the logit. Purely generalized points are more robust: more noise is needed to flip the classification.

We sweep over several epsilon values on the calibration set and select the one that gives the highest AUC. We then use it on the test set.

In [ ]:
N_DIRS = 1000

def perturbate(eps, X):
    flips = torch.zeros(len(X), device=X.device)
    with torch.no_grad():
        orig_preds = (model(X).squeeze() > 0).float()
        for _ in range(N_DIRS):
            noise = torch.randn_like(X) * eps
            pert_preds = (model(X + noise).squeeze() > 0).float()
            flips += (orig_preds != pert_preds).float()
    return flips / N_DIRS

EPS_GRID = [0.001, 0.005, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07,
            0.08, 0.09, 0.10, 0.13, 0.16, 0.20, 0.25, 0.30, 0.40, 0.50]

best_eps, best_auc = 0, 0
for eps in EPS_GRID:
    flip_rates = perturbate(eps, X_calibration).cpu()
    auc = roc_auc_score(y_calibration.cpu(), flip_rates)
    auc = max(auc, 1 - auc)
    if auc > best_auc:
        best_eps, best_auc = eps, auc

print(f'best eps on calibration set: {best_eps} (AUC={best_auc:.4f})')
evaluate(perturbate(best_eps, X_test).cpu(), f'Perturbation Flip Rate (eps={best_eps})')

### 2. **Logreg on `[logit, logit − average perturbed logit]` features**

Two-component feature: (i) the logit itself, (ii) the difference between the original and the average logit in a noisy environment. The latter measures how stable the network's response is. We weight and combine them with logistic regression on the calibration set, then apply on the test set.

In [ ]:
def extract_features(X_tensor):
    with torch.no_grad():
        orig_logits = model(X_tensor).squeeze()
        avg_pert_logits = torch.zeros_like(orig_logits)
        for _ in range(N_DIRS):
            noise = torch.randn_like(X_tensor) * best_eps
            avg_pert_logits += model(X_tensor + noise).squeeze()
        avg_pert_logits /= N_DIRS
        f1 = orig_logits.cpu().numpy()
        f2 = (orig_logits - avg_pert_logits).cpu().numpy()
    return np.stack([f1, f2], axis=1)

X_train_lr = extract_features(X_calibration)
X_test_lr  = extract_features(X_test)

clf = LogisticRegression()
clf.fit(X_train_lr, y_calibration.cpu())

probs = clf.predict_proba(X_test_lr)[:, 1]
evaluate(probs, 'Logreg [logit, logit−avg_pert]')

### 3. **Input Gradient Norm**

The trained model's gradient with respect to the input is typically larger on memorized points because the network learned a sharper decision surface there. Simple, fast signal, does not require perturbation or another model.

In [ ]:
test_in = X_test.clone().requires_grad_(True)
out_logits = model(test_in).squeeze()
grads = torch.autograd.grad(outputs=out_logits.sum(), inputs=test_in)[0]
grad_norms = grads.norm(dim=1).detach().cpu().numpy()

evaluate(grad_norms, 'Input Gradient Norm')

### 4. **Activation Probe**

We feed the activations of the last hidden layer into logistic regression with calibration labels, then apply to the test set. The logreg uses `class_weight='balanced'` due to the 33/17 calibration imbalance, and `C=0.1` to avoid overfitting on the small training set.

In [ ]:
with torch.no_grad():
    def hidden(x):
        x = torch.sin(30 * model.fc1(x))
        return torch.sin(30 * model.fc2(x))

    cal_acts  = hidden(X_calibration).cpu().numpy()
    test_acts = hidden(X_test).cpu().numpy()

probe = LogisticRegression(class_weight='balanced', max_iter=1000, C=0.1)
probe.fit(cal_acts, y_calibration.cpu())
probe_probs = probe.predict_proba(test_acts)[:, 1]

evaluate(probe_probs, 'Activation Probe')

### 5. **Distillation Disagreement**

We train a *clean* model on the trained model's outputs on **fresh, random points**; these contain no memorized labels, only the learned function. The disagreement between the clean model and the original model on the test set is larger for memorized points, because the original network learned label leakage there which the clean model does not reproduce.

This is the most computationally intensive strategy: generating 50k synthetic points + 5000 steps of Adam training (approx. 1 minute on GPU).

In [ ]:
from copy import deepcopy

n_fresh = 50_000
fresh_X = torch.rand(n_fresh, 12, device=device) * 2 - 1

with torch.no_grad():
    fresh_y = model(fresh_X).squeeze()

clean_model = deepcopy(model)
with torch.no_grad():
    clean_model.fc1.weight.uniform_(-1 / 12, 1 / 12)
    bound2 = (6 / 256) ** 0.5 / 30
    clean_model.fc2.weight.uniform_(-bound2, bound2)
    bound3 = (6 / 256) ** 0.5
    clean_model.fc3.weight.uniform_(-bound3, bound3)

optimizer = torch.optim.Adam(clean_model.parameters(), lr=1e-4)
for step in range(5000):
    pred = clean_model(fresh_X).squeeze()
    loss = ((pred - fresh_y) ** 2).mean()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (step + 1) % 1000 == 0:
        print(f'  step {step+1:5d}  loss={loss.item():.6f}')

with torch.no_grad():
    orig_out  = model(X_test).squeeze()
    clean_out = clean_model(X_test).squeeze()

disagreement = (orig_out - clean_out).abs().cpu().numpy()
evaluate(disagreement, 'Distillation Disagreement')

## **Lessons learned**

- **Baseline logit confidence already gives a significant AUC**, because the flipped points received higher sample weight during training, pushing them close to the boundary.
- **Perturbation flip rate** is the classical signal that directly exploits the sharp decision boundary of memorized points. The epsilon must be calibrated on the calibration set (`best_eps`).
- **Logreg fusion** (original logit + flip-rate-like features) often performs best, because the two components are complementary (one measures confidence, the other stability).
- **Input gradient norm + activation probe** are fast and cheap alternatives; their ROC AUC typically approaches that of the perturbation method.
- **Distillation disagreement** is computationally expensive, but theoretically the cleanest signal: sensitive only to memorization, not to the learned function.

Due to the convex nature of the scoring curve, combining strategies (e.g., weighted average, or a meta-logreg) in the upper range (0.9 → 0.95 → 1.0) can yield significantly more points than the best individual strategy.